In [13]:
# Libraries
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [14]:
# Loading
data = pd.read_csv("spotify_history_multiuser.csv")
print(data.head())

# data exploring
data.info()
print("\n")
print(data.describe())

                    ts    track_name     artist_name  \
0  2013-07-10 08:30:00  Moth's Wings     Passion Pit   
1  2015-01-12 22:21:00            XO      John Mayer   
2  2015-01-12 22:25:00     Mr. Jones  Counting Crows   
3  2015-08-05 23:55:00   Working Man            Rush   
4  2015-08-06 06:55:00            XO      John Mayer   

                    album_name  ms_play reason_start reason_end  shuffle  \
0                      Manners   147536      unknown    unknown    False   
1                           XO    31227      appload  trackdone    False   
2  August And Everything After   272506    trackdone  trackdone    False   
3                         Rush    43353     clickrow    endplay     True   
4                           XO    43670     clickrow    endplay     True   

   skipped  user_id  replayed_within_30_days  
0     True  user_01                        0  
1    False  user_01                        0  
2    False  user_01                        0  
3     True  user_0

In [15]:
# data cleaning
print("Missing values:\n", data.isnull().sum())
data = data.dropna()
data = data.drop_duplicates()

Missing values:
 ts                         0
track_name                 0
artist_name                0
album_name                 0
ms_play                    0
reason_start               2
reason_end                 3
shuffle                    0
skipped                    0
user_id                    0
replayed_within_30_days    0
dtype: int64


In [16]:
# data preprocessing
data['ts'] = pd.to_datetime(data['ts'])  # convert timestamp column to datetime
data = data.sort_values(['user_id', 'track_name', 'ts']).reset_index(drop=True)

# extract useful features from the timestamp
data['hour'] = data['ts'].dt.hour
data['day_of_week'] = data['ts'].dt.dayofweek

# how many times this user has played this track before (so far)
data['play_count_so_far'] = data.groupby(['user_id', 'track_name']).cumcount()

# convert True/False columns to 0/1 so the model can use them
data['shuffle'] = data['shuffle'].astype(int)
data['skipped'] = data['skipped'].astype(int)

In [17]:
# Encode text categories into numbers
encoder = LabelEncoder()
data['reason_start'] = encoder.fit_transform(data['reason_start'].astype(str))
data['reason_end'] = encoder.fit_transform(data['reason_end'].astype(str))
data['user_id_enc'] = encoder.fit_transform(data['user_id'])


In [18]:
# Features x and y
X = data[['ms_play', 'shuffle', 'skipped', 'hour', 'day_of_week',
          'play_count_so_far', 'reason_start', 'reason_end', 'user_id_enc']]
y = data['replayed_within_30_days']

In [19]:
# Normalize features to a 0-1 range
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

# model train and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [20]:
# Model 1: Logistic Regression
model_lr = LogisticRegression()
model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)

print("Logistic Regression:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_lr),4))
print("Precision: ", round(precision_score(y_test, y_pred_lr),4))
print("Recall:", round(recall_score(y_test, y_pred_lr),4))
print("F1-Score:", round(f1_score(y_test, y_pred_lr),4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

# Model 2: Decision Tree
model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train, y_train)
y_pred_dt = model_dt.predict(X_test)

print("\nDecision Tree:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_dt),4))
print("Precision: ", round(precision_score(y_test, y_pred_dt),4))
print("Recall:", round(recall_score(y_test, y_pred_dt),4))
print("F1-Score:", round(f1_score(y_test, y_pred_dt),4))

Logistic Regression:
Accuracy: 0.64
Precision:  0.6364
Recall: 0.2333
F1-Score: 0.3415
Confusion Matrix:
 [[164  16]
 [ 92  28]]

Decision Tree:
Accuracy: 0.5667
Precision:  0.4576
Recall: 0.45
F1-Score: 0.4538


In [21]:
# Model 3: Random Forest
model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

print("\nRandom Forest")
print("Accuracy:", round(accuracy_score(y_test, y_pred_rf),4))
print("Precision: ", round(precision_score(y_test, y_pred_rf),4))
print("Recall:", round(recall_score(y_test, y_pred_rf),4))
print("F1-Score:", round(f1_score(y_test, y_pred_rf),4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

# Model 4: KNN
model_knn = KNeighborsClassifier()
model_knn.fit(X_train, y_train)
y_pred_knn = model_knn.predict(X_test)

print("\nKNN")
print("Accuracy:", round(accuracy_score(y_test, y_pred_knn),4))
print("Precision: ", round(precision_score(y_test, y_pred_knn),4))
print("Recall:", round(recall_score(y_test, y_pred_knn),4))
print("F1-Score:", round(f1_score(y_test, y_pred_knn),4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_knn))


Random Forest
Accuracy: 0.63
Precision:  0.5506
Recall: 0.4083
F1-Score: 0.4689
Confusion Matrix:
 [[140  40]
 [ 71  49]]

KNN
Accuracy: 0.6067
Precision:  0.5119
Recall: 0.3583
F1-Score: 0.4216
Confusion Matrix:
 [[139  41]
 [ 77  43]]


In [22]:
# Predict probability for every track using the best model (Random Forest)
data['predicted_replay_prob'] = model_rf.predict_proba(X)[:, 1]

# Showing top 5 recommended tracks for each user
for user in data['user_id'].unique():
    top_tracks = data[data['user_id'] == user].sort_values(
        'predicted_replay_prob', ascending=False
    )[['track_name', 'artist_name', 'predicted_replay_prob']].head(5)

    print(f"\nTop 5 recommended tracks for {user}:")
    print(top_tracks)


Top 5 recommended tracks for user_01:
                                            track_name      artist_name  \
82                                        I’m So Sorry  Imagine Dragons   
18                                      Beyond the Sea      Bobby Darin   
125                                       Read My Mind      The Killers   
85   Kansas City / Hey-Hey-Hey-Hey - Medley / Remas...      The Beatles   
112                                       No Surprises        Radiohead   

     predicted_replay_prob  
82                    0.98  
18                    0.95  
125                   0.95  
85                    0.95  
112                   0.95  

Top 5 recommended tracks for user_02:
                                track_name          artist_name  \
227                         Cascos Ligeros  Alejandro Fernández   
300                           Losing Touch          The Killers   
364                        Time to Pretend                 MGMT   
241  Don't Look Back in Anger